In [5]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
!git clone https://github.com/seungil-kim/VehLatForceEstimation.git
%cd VehLatForceEstimation

fatal: destination path 'VehLatForceEstimation' already exists and is not an empty directory.
/content/VehLatForceEstimation


In [2]:
## 데이터/라이브러리 불러오기 ##
import numpy as np
from pathlib import Path
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [14]:
## 데이터셋 불러오기 ##
data_path = Path("/content/drive/MyDrive/LSTM_LatForce/data")
mat_file = data_path / "LSTM_Dataset.mat"
mat_data = loadmat(mat_file)
print(mat_data.keys())
dataset = mat_data["Dataset"]
print(type(dataset))
print(dataset.shape)

dict_keys(['__header__', '__version__', '__globals__', 'Dataset'])
<class 'numpy.ndarray'>
(1, 1)


In [28]:
## 데이터셋 branch ##
scenario_names = dataset.dtype.names

for name in scenario_names:
    scenario = dataset[0, 0][name]

    X = scenario["X"][0, 0]
    Y = scenario["Y"][0, 0]

    print(name, "X:", X.shape, "Y:", Y.shape)

Step_Lturn_mu0p85_v40 X: (751, 6) Y: (751, 2)
Step_Rturn_mu0p85_v40 X: (751, 6) Y: (751, 2)
Step_Lturn_mu0p85_v60 X: (751, 6) Y: (751, 2)
Step_Rturn_mu0p85_v60 X: (751, 6) Y: (751, 2)
SLC_Lturn_mu0p85_v40 X: (677, 6) Y: (677, 2)
SLC_Rturn_mu0p85_v40 X: (677, 6) Y: (677, 2)
SLC_Lturn_mu0p85_v60 X: (452, 6) Y: (452, 2)
SLC_Rturn_mu0p85_v60 X: (452, 6) Y: (452, 2)
OnCenterSteer_mu0p85_v40 X: (4508, 6) Y: (4508, 2)
OnCenterSteer_mu0p85_v60 X: (3005, 6) Y: (3005, 2)
Urban_mu0p85 X: (18106, 6) Y: (18106, 2)


In [30]:
def seq_data(x, y, sequence_length):
    x_seq = []
    y_seq = []

    for i in range(len(x) - sequence_length + 1):
        x_seq.append(x[i : i + sequence_length])
        y_seq.append(y[i + sequence_length - 1])

    return np.array(x_seq), np.array(y_seq)

In [38]:
sequence_length = 5

scenario_names = dataset.dtype.names

x_train_list = []
y_train_list = []

for name in scenario_names:
    scenario = dataset[0, 0][name]

    X = scenario["X"][0, 0]
    Y = scenario["Y"][0, 0]

    x_seq, y_seq = seq_data(X, Y, sequence_length)

    x_train_list.append(x_seq)
    y_train_list.append(y_seq)

    print(f"{name} -> X_seq: {x_seq.shape}, Y_seq: {y_seq.shape}")

x_train_seq = np.concatenate(x_train_list, axis=0)
y_train_seq = np.concatenate(y_train_list, axis=0)

print("================================")
print("Train X:", x_train_seq.shape)
print("Train Y:", y_train_seq.shape)

Step_Lturn_mu0p85_v40 -> X_seq: (747, 5, 6), Y_seq: (747, 2)
Step_Rturn_mu0p85_v40 -> X_seq: (747, 5, 6), Y_seq: (747, 2)
Step_Lturn_mu0p85_v60 -> X_seq: (747, 5, 6), Y_seq: (747, 2)
Step_Rturn_mu0p85_v60 -> X_seq: (747, 5, 6), Y_seq: (747, 2)
SLC_Lturn_mu0p85_v40 -> X_seq: (673, 5, 6), Y_seq: (673, 2)
SLC_Rturn_mu0p85_v40 -> X_seq: (673, 5, 6), Y_seq: (673, 2)
SLC_Lturn_mu0p85_v60 -> X_seq: (448, 5, 6), Y_seq: (448, 2)
SLC_Rturn_mu0p85_v60 -> X_seq: (448, 5, 6), Y_seq: (448, 2)
OnCenterSteer_mu0p85_v40 -> X_seq: (4504, 5, 6), Y_seq: (4504, 2)
OnCenterSteer_mu0p85_v60 -> X_seq: (3001, 5, 6), Y_seq: (3001, 2)
Urban_mu0p85 -> X_seq: (18102, 5, 6), Y_seq: (18102, 2)
Train X: (30837, 5, 6)
Train Y: (30837, 2)


In [39]:
## GPU 사용 여부 확인 (Cuda)
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print("▶device: ", device)

# 텐서로 변환
x_train_seq = torch.FloatTensor(x_train_seq).to(device)
y_train_seq = torch.FloatTensor(y_train_seq).to(device)

print(x_train_seq.size())
print(y_train_seq.size())

▶device:  cuda:0
torch.Size([30837, 5, 6])
torch.Size([30837, 2])


In [ ]:
## 배치 형태로 변경 ##

train_dataset = torch.utils.data.TensorDataset(x_train_seq, y_train_seq)
#test_dataset = torch.utils.data.TensorDataset(x_test_seq, y_test_seq)

batch_size = 128
train_loader = torch.utils.data.DataLoader(dataset=train_dataset, batch_size = batch_size, shuffle=True)
#test_loader = torch.utils.data.DataLoader(dataset=test_dataset, batch_size = batch_size)